# Phishing URL Detection Using Machine Learning
**Student:** Shruti Tukaram Avhad  
**Program:** IBM SkillsBuild Cyber Security 2026 — BharatCares  
**Project type:** Educational cybersecurity / machine learning

## Project overview
This notebook trains a machine-learning model to classify URLs as phishing/suspicious or legitimate using character-level TF-IDF features and Logistic Regression. It evaluates the model with a held-out test set and reports accuracy, precision, recall, F1-score, and a confusion matrix.

**Important:** This is a learning demonstration, not a production security control. A URL classifier can make false predictions. Do not open suspicious URLs to test them; classification is performed on the URL text only.

## 1. Install and import dependencies
Run this notebook in Jupyter or Google Colab. If the libraries are not installed, run the installation cell first.

In [ ]:
# If needed, uncomment and run:
# %pip install -r requirements.txt

import re
import warnings
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load the dataset
The primary dataset URL below points to a public CSV commonly used for malicious-URL classification. The notebook tries to load it automatically. If the source is unavailable or its format has changed, upload a CSV containing a URL column and a label column, or set `DATASET_PATH` to a local file.

Expected columns include a URL field (`url`) and a class field (`label`). The code attempts to identify common alternatives.

In [ ]:
DATASET_URL = (
    "https://raw.githubusercontent.com/faizann24/"
    "Using-machine-learning-to-detect-malicious-URLs/master/data/data.csv"
)
DATASET_PATH = None  # Example: "data.csv" if you have downloaded the dataset

def load_dataset():
    if DATASET_PATH:
        return pd.read_csv(DATASET_PATH)
    return pd.read_csv(DATASET_URL)

try:
    df = load_dataset()
    print("Dataset loaded:", df.shape)
    display(df.head())
except Exception as exc:
    print("Automatic dataset download failed:", exc)
    print("Upload a CSV with URL and label columns, then set DATASET_PATH to its path.")
    raise RuntimeError(
        "Dataset unavailable. Please download the dataset or upload a compatible CSV."
    )

## 3. Prepare URL and label columns
Labels vary between datasets. The preparation step maps common labels such as `good`, `benign`, `legitimate`, `bad`, `phishing`, and `malicious` to two classes. The model uses **0 = legitimate** and **1 = phishing/malicious**.

In [ ]:
def find_column(columns, candidates):
    lowered = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    return None

url_col = find_column(df.columns, ["url", "uri", "website", "link"])
label_col = find_column(df.columns, ["label", "type", "class", "status", "result"])

if url_col is None or label_col is None:
    raise ValueError(
        f"Could not identify URL and label columns. Available columns: {list(df.columns)}"
    )

data = df[[url_col, label_col]].copy()
data.columns = ["url", "raw_label"]
data = data.dropna()
data["url"] = data["url"].astype(str).str.strip()
data["raw_label"] = data["raw_label"].astype(str).str.strip().str.lower()
data = data[data["url"].str.len() > 3]

positive_labels = {
    "bad", "phishing", "malicious", "malware", "defacement", "1",
    "phish", "suspicious", "fraud"
}
negative_labels = {
    "good", "benign", "legitimate", "safe", "normal", "0",
    "clean", "legit"
}

def map_label(value):
    value = str(value).strip().lower()
    if value in positive_labels:
        return 1
    if value in negative_labels:
        return 0
    # Numeric labels: treat 1 as positive and 0 as negative only.
    return np.nan

data["label"] = data["raw_label"].map(map_label)
data = data.dropna(subset=["label"])
data["label"] = data["label"].astype(int)
data = data.drop_duplicates(subset=["url"])
data = data[data["label"].isin([0, 1])]

if data["label"].nunique() != 2:
    raise ValueError(
        "The dataset did not produce both classes after label mapping. "
        "Inspect the label values and update positive_labels/negative_labels."
    )

print("Usable rows:", len(data))
print("Class counts (0=legitimate, 1=phishing/malicious):")
display(data["label"].value_counts().rename(index={0: "Legitimate", 1: "Phishing/malicious"}))
display(data.head())

## 4. Split the data and train the model
Character-level TF-IDF can learn useful patterns in URL strings without visiting the websites. A stratified split keeps the class proportions similar in training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["url"],
    data["label"],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=data["label"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_features=200_000,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
print("Training completed.")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Evaluate the model
Accuracy alone can be misleading when classes are imbalanced. Review precision, recall, F1-score, and the confusion matrix as well.

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Legitimate", "Phishing/malicious"],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
display(pd.DataFrame(
    cm,
    index=["Actual legitimate", "Actual phishing/malicious"],
    columns=["Predicted legitimate", "Predicted phishing/malicious"]
))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate", "Phishing/malicious"]
).plot(values_format="d")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 6. Test a URL string
Enter a URL as text. The model does not connect to or open the URL. Its prediction is only an indication and must not be treated as a guarantee of safety.

In [ ]:
def classify_url(url_text):
    url_text = str(url_text).strip()
    if not url_text:
        raise ValueError("Please provide a non-empty URL string.")
    prediction = int(model.predict([url_text])[0])
    label = "Potentially phishing/malicious" if prediction == 1 else "Likely legitimate"
    confidence = None
    if hasattr(model, "predict_proba"):
        confidence = float(model.predict_proba([url_text]).max())
    return {"url": url_text, "prediction": label, "model_confidence": confidence}

# Change this example to a URL string you want to inspect.
# Do not visit suspicious URLs in a browser.
example_url = "https://example.com/login"
classify_url(example_url)

## 7. Save the trained model (optional)
This saves the pipeline for later use. Only load model files from trusted sources.

In [ ]:
import joblib

MODEL_OUTPUT = "phishing_url_model.joblib"
joblib.dump(model, MODEL_OUTPUT)
print(f"Model saved to: {MODEL_OUTPUT}")

## 8. Limitations and ethical use
- A model can miss new phishing campaigns or incorrectly flag safe websites.
- Performance depends on dataset quality, label quality, class balance, and how representative the test data is.
- URL text alone cannot establish whether a website is safe.
- Use only public, authorized datasets. Do not visit, interact with, or distribute harmful URLs.
- For real-world protection, combine multiple security signals and use updated threat intelligence.

## Conclusion
This project demonstrates a basic machine-learning workflow for URL classification: dataset preparation, feature extraction, model training, evaluation, and prediction. The evaluation results should be recorded from the actual notebook run rather than assumed in advance.